In [1]:
%pip install ucimlrepo --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
ds = fetch_ucirepo(id=601) # 601 = AI4I 2020 Predictive Maintenance
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (10000, 12)
Columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
df = df_raw.copy()
df.columns = [c.split(" [")[0].strip() for c in df.columns]
rename_map = {
"Type": "type",
"Air temperature": "air_temp_k",
"Process temperature": "process_temp_k",
"Rotational speed": "rpm",
"Torque": "torque_nm",
"Tool wear": "tool_wear_min",
"Machine failure": "machine_failure",
}
df = df.rename(columns=rename_map)[list(rename_map.values())]

In [4]:
print("Missing values per column:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nTarget balance (share of rows):")
print(df["machine_failure"].value_counts(normalize=True).round(4))

Missing values per column:
type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64

Data types:
type                object
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Target balance (share of rows):
0    0.9661
1    0.0339
Name: machine_failure, dtype: float64


In [5]:
df.to_csv("../data/ai4i_clean.csv", index=False)
print("Saved", df.shape, "to data/ai4i_clean.csv")

Saved (10000, 7) to data/ai4i_clean.csv


In [12]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
SUBSCRIPTION_ID = "2e093406-ee7d-40a3-a0a5-8abd8931cead"
RESOURCE_GROUP = "rg-mlab1-fatimakhan"
WORKSPACE = "mlw-lab1-fatimakhan"
ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Connected to: mlw-lab1-fatimakhan


In [14]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
csv_asset = Data(
name="ai4i-maintenance-csv",
version="1",
type=AssetTypes.URI_FILE,
path="../data/ai4i_clean.csv",
description="AI4I 2020 predictive maintenance, cleaned: 6 features +machine_failure",
)
csv_asset = ml_client.data.create_or_update(csv_asset)
print("Registered:", csv_asset.name, "version", csv_asset.version)

Uploading ai4i_clean.csv (< 1 MB): 100%|██████████| 295k/295k [00:00<00:00, 8.17MB/s]




Registered: ai4i-maintenance-csv version 1


In [1]:
import mltable
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

# 1. Create the MLTable definition from your local CSV
paths = [{'file': '../data/ai4i_clean.csv'}]
tbl = mltable.from_delimited_files(paths)
tbl.save('../data/')  # Saves the 'MLTable' metadata file in the data folder

# 2. Register it directly in your Azure ML Workspace
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
mltable_asset = Data(
    name="ai4i-maintenance-mltable",
    version="1",
    type=AssetTypes.MLTABLE,
    path="../data/",
    description="MLTable asset for AutoML"
)
ml_client.data.create_or_update(mltable_asset)
print("MLTable successfully registered!")

Found the config file in: /config.json
Uploading data (0.3 MBs): 100%|██████████| 295579/295579 [00:00<00:00, 5432131.37it/s]




MLTable successfully registered!
